In [ ]:
import pandas as pd
import plotly.graph_objects as go
from sqlalchemy import text
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src' / 'amal'))
from database import get_engine

engine = get_engine()

# Load labeled data from DB
df = pd.read_sql("SELECT * FROM bubble_analysis WHERE ticker_id='NIFTY50' ORDER BY date", engine)
market = pd.read_sql("SELECT date, close FROM market_data WHERE ticker_id='NIFTY50' ORDER BY date", engine)

df = df.merge(market, on="date")

# Plot
fig = go.Figure()

# Price line
fig.add_trace(go.Scatter(x=df["date"], y=df["close"], name="NIFTY Price", line=dict(color="black", width=1.5)))

# Bubble periods — red shading
bubbles = df[df["label"] == "Bubble"]
fig.add_trace(go.Scatter(x=bubbles["date"], y=bubbles["close"], mode="markers", name="Bubble", marker=dict(color="red", size=4)))

# Crash periods — blue shading
crashes = df[df["label"] == "Crash"]
fig.add_trace(go.Scatter(x=crashes["date"], y=crashes["close"], mode="markers", name="Crash", marker=dict(color="blue", size=4)))

fig.update_layout(title="NIFTY 50 — Bubble & Crash Labels", xaxis_title="Date", yaxis_title="Price")
fig.show()